# 01 — Data collection (simulated)

**This is not real store data.** We generated it so we already know true demand. That lets us *see* the censoring problem. It does not prove a method works in a shop.

Rule:

`sales = min(true_demand, opening_stock)`

Two files:

- `data/raw/observed_daily.csv` — what the store could log
- `data/raw/with_ground_truth.csv` — plus hidden demand (**evaluation only**, never a training feature)


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(
    p for p in (Path("."), Path(".."))
    if (p / "data" / "raw" / "observed_daily.csv").exists()
)
observed = pd.read_csv(ROOT / "data" / "raw" / "observed_daily.csv")
truth = pd.read_csv(ROOT / "data" / "raw" / "with_ground_truth.csv")

print("observed", observed.shape)
print("truth   ", truth.shape)
print("data_source values:", observed["data_source"].unique())
print("products:", sorted(observed["product"].unique()))
print("date range:", observed["date"].min(), "→", observed["date"].max())

## What a store would actually record

`pd.read_csv` loads the file into a DataFrame (a table). `head()` shows the first rows. There is **no** `true_demand` column here.


In [ ]:
observed.head(8)

## The hidden column

Compare `true_demand` vs `sales` on Biscuits. When stock is 8 and 14 people wanted biscuits, the log still says `sales = 8`.


In [ ]:
biscuits = truth[truth["product"] == "Biscuits"][
    ["date", "opening_stock", "true_demand", "sales", "closing_stock", "is_censored"]
]
biscuits.head(14)

Read Sep 1–3 like this:

| date | stock | true demand | recorded sales |
| --- | ---: | ---: | ---: |
| Sep 1 | 8 | 14 | 8 |
| Sep 2 | 0 | 8 | 0 |
| Sep 3 | 0 | 5 | 0 |

A naive model that only sees sales learns `8, 0, 0` and can conclude demand collapsed. It did not.


In [ ]:
summary = (
    truth.groupby("product", as_index=False)
    .agg(
        days=("date", "count"),
        sales_units=("sales", "sum"),
        true_demand_units=("true_demand", "sum"),
        lost_sales_units=("lost_sales", "sum"),
        censored_days=("is_censored", "sum"),
        stockout_flag_days=("stockout_flag", "sum"),
    )
    .sort_values("censored_days", ascending=False)
)
summary

## One chart: sales is not demand

If sales drop whenever inventory hits zero, you are looking at **censoring**, not a demand crash.


In [ ]:
plot_df = biscuits.copy()
plot_df["date"] = pd.to_datetime(plot_df["date"])
plot_df = plot_df.head(30)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(plot_df["date"], plot_df["true_demand"], label="true demand (hidden)")
ax.plot(plot_df["date"], plot_df["sales"], label="observed sales")
ax.plot(plot_df["date"], plot_df["opening_stock"], label="opening stock", linestyle="--")
ax.set_title("Biscuits — first 30 days (simulated)")
ax.set_ylabel("units")
ax.legend()
fig.autofmt_xdate()
plt.show()

## Leakage rule (write this down)

Do **not** train on `true_demand`, `lost_sales`, `is_censored`, or `demand_mean`. A real store never sees those. We keep them so later we can test whether a stockout-aware model recovers demand better than one trained on `sales` alone.

Next phase: pandas on *this* table (missing values, dtypes, duplicates) — still no ML.
